# Local Agentic AI Workshop (Offline)
## Build an Agentic AI assistant + web chat UI on your laptop (no cloud, no API keys)

**What you will build**
- Local LLM chat function (via Ollama)
- Minimal agent loop (PLAN → TOOL → FINAL)
- Local tools (calculator, checklist, pros/cons, time blocks)
- Local memory saved to `memory.json`
- A Streamlit web chat UI

Run this notebook top-to-bottom, or section-by-section.


In [1]:
import sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())


Python: 3.14.3 (v3.14.3:323c59a5e34, Feb  3 2026, 11:41:37) [Clang 16.0.0 (clang-1600.0.26.6)]
Platform: macOS-15.3.1-arm64-arm-64bit-Mach-O


In [2]:
# If needed (run once):
# !pip install -r requirements.txt


In [3]:
import requests
r = requests.get('http://localhost:11434/api/tags', timeout=10)
print('Ollama reachable ✅')
print('Available models:', [m['name'] for m in r.json().get('models', [])][:10])


Ollama reachable ✅
Available models: ['llama3.2:1b']


In [4]:
import requests

def ollama_chat(messages, model='llama3.2:1b', temperature=0.2, base_url='http://localhost:11434'):
    url = f'{base_url}/api/chat'
    payload = {'model': model, 'messages': messages, 'options': {'temperature': temperature}, 'stream': False}
    resp = requests.post(url, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()['message']['content']

messages = [
  {'role': 'system', 'content': 'You are friendly and concise.'},
  {'role': 'user', 'content': 'Say hello in one sentence and ask what I want to plan.'}
]
print(ollama_chat(messages))


Hello! What do you want to plan today?


In [5]:
history = [{'role':'system','content':'You are friendly and concise.'}]

def chat_turn(user_text, model='llama3.2:1b'):
    history.append({'role':'user','content': user_text})
    reply = ollama_chat(history, model=model)
    history.append({'role':'assistant','content': reply})
    return reply

print(chat_turn('I want to plan my weekend.'))
print(chat_turn('Budget is $50 and I like vegetarian food.'))


I'd be happy to help you plan your weekend. Can you tell me a bit more about what you're looking for? Do you have any specific activities or events planned, or are you open to suggestions? Are you looking for something relaxing and low-key, or are you up for something more adventurous?
With a budget of $50 and a preference for vegetarian food, here are a few ideas for a fun and relaxing weekend:

1. **Picnic in the park**: Pack a basket with some of your favorite vegetarian snacks and head to a nearby park for a relaxing afternoon in the sun. You can find a nice spot under a tree or by a lake, and enjoy the scenery. (Free)
2. **Cooking at home**: Look up some easy and delicious vegetarian recipes online, and head to the grocery store to pick up the ingredients. You can make a big pot of pasta, stir-fry, or roast some vegetables for a tasty and satisfying meal. (Approx. $20 for ingredients)
3. **Visit a local farmer's market**: Many cities have weekly farmer's markets where you can find

In [6]:
from tools import TOOLS_REGISTRY, list_tools
print('Available tools:', list_tools())
print('Calculator example:', TOOLS_REGISTRY['calculator']('(1200/12)*0.8'))


Available tools: ['calculator', 'checklist', 'pros_cons', 'timeblock_plan']
Calculator example: 80.0


In [7]:
import json
from agent import build_tool_selection_messages, parse_tool_call, execute_tool, build_final_messages, load_memory

memory = load_memory('memory.json')
user_text = 'Create a checklist for hosting a kids birthday party for 10 people.'

tool_messages = build_tool_selection_messages(user_text, memory)
decision = ollama_chat(tool_messages, model='llama3.2:1b', temperature=0.0)
print('Tool decision raw:')
print(decision)

tool, args = parse_tool_call(decision)
print('\nParsed tool:', tool, args)

tool_result = execute_tool(tool, args) if tool else None
print('\nTool result:')
print(tool_result)

final_messages = build_final_messages(user_text, tool, tool_result, memory)
final_answer = ollama_chat(final_messages, model='llama3.2:1b', temperature=0.2)
print('\nFinal answer:')
print(final_answer)


ImportError: cannot import name 'build_tool_selection_messages' from 'agent' (/Users/vinay/PracticeFolder/agentic-local-workshop/agent.py)

In [ ]:
from agent import load_memory, save_memory, maybe_update_memory
memory = load_memory('memory.json')
print('Before:', memory['profile'])
maybe_update_memory('My name is Vinay', memory)
maybe_update_memory('Remember that I prefer vegetarian meals', memory)
save_memory(memory, 'memory.json')
print('After:', load_memory('memory.json')['profile'])


In [ ]:
from agent import agent_reply
print(agent_reply('Plan a fun Saturday for me under $50. Ask clarifying questions if needed.'))


## Streamlit UI
Run this in a terminal (inside this folder):

```bash
streamlit run app.py
```

Then open: http://localhost:8501
